# 6.4 LDA

In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import gensim
import gensim.corpora as corpora

### Load Data

In [2]:
data = pd.read_csv("data/news_articles.csv") # data should be in the same folder as your notebook

In [3]:
data.head()

,id,title,content
0,25626,"One Weight-Loss Approach Fits All? No, Not Eve...","Dr. Frank Sacks, a professor of nutrition at H..."
1,19551,South Carolina Stuns Baylor to Reach the Round...,South Carolina’s win over Duke was not only ...
2,25221,"U.S. Presidential Race, Apple, Gene Wilder: Yo...",(Want to get this briefing by email? Here’s th...
3,18026,"His Predecessor Gone, Gambia’s New President F...","BANJUL, Gambia — A week after he was inaugu..."
4,21063,‘Harry Potter and the Cursed Child’ Goes From ...,The biggest book of the summer isn’t a blockbu...


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       100 non-null    int64 
 1   title    100 non-null    object
 2   content  100 non-null    object
dtypes: int64(1), object(2)
memory usage: 2.5+ KB


### Clean Data

In [5]:
# take just the content of the article, lowercase and remove punctuation
articles = data['content'].str.lower().apply(lambda x: re.sub(r"([^\w\s])", "", x))

# stop word removal
en_stopwords = stopwords.words('english')
articles = articles.apply(lambda x: ' '.join([word for word in x.split() if word not in (en_stopwords)]))

# tokenize
articles = articles.apply(lambda x: word_tokenize(x))

# stemming (done for speed as we have a lot of text)
ps = PorterStemmer()
articles = articles.apply(lambda tokens: [ps.stem(token) for token in tokens])

In [6]:
articles

0     [dr, frank, sack, professor, nutrit, harvard, ...
1     [south, carolina, win, duke, surpris, fan, pos...
2     [want, get, brief, email, here, good, even, he...
3     [banjul, gambia, week, inaugur, anoth, countri...
4     [biggest, book, summer, isnt, blockbust, thril...
                            ...                        
95    [want, get, brief, email, here, good, even, he...
96    [tallinn, estonia, guard, brought, ahm, abdul,...
97    [gov, scott, walker, wisconsin, activ, wiscons...
98    [social, media, shook, emot, headlin, shout, n...
99    [moment, joanna, acevedo, first, set, foot, bo...
Name: content, Length: 100, dtype: object

### Vectorization

In [22]:
# create dictionary of all words
dictionary = corpora.Dictionary(articles)
print(dictionary)

Dictionary<8693 unique tokens: ['10', '100', '108', '15', '155']...>


In [8]:
# vecotize using bag of words into a document term matrix
doc_term = [dictionary.doc2bow(text) for text in articles]

### LDA

In [16]:
# specify number of topics
num_topics = 2

In [17]:
# create LDA model
lda_model = gensim.models.LdaModel(corpus=doc_term,
                                   id2word=dictionary,
                                   num_topics=num_topics)

In [12]:
lda_model.print_topics(num_topics=num_topics, num_words=5)

[(0,
  '0.020*"mr" + 0.015*"said" + 0.006*"trump" + 0.005*"state" + 0.005*"would"'),
 (1, '0.014*"said" + 0.012*"mr" + 0.005*"year" + 0.005*"one" + 0.004*"trump"')]

## Alternative Approach

In [29]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

In [43]:
df = pd.read_csv("data/news_articles.csv")
df = pd.read_csv("data/npr.csv")

In [45]:
df.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


In [46]:
cv = CountVectorizer(max_df=0.9, min_df=2, stop_words="english")
dtm = cv.fit_transform(df["Article"])
lda = LatentDirichletAllocation(n_components=7, random_state=42)
lda.fit(dtm)

,n_components,7
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


We can obtain the number of tokens in our corpus as follows

In [40]:
len(cv.get_feature_names_out()) 

5414

The components property is a list of length k where k is the number of topics. Every element in the list is another list which contains the probability that a word belongs to the topic.

In [35]:
len(lda.components_[0])

5414

In [37]:
n = 15
for index, topic in enumerate(lda.components_):
    print(f'The top {n} words for topic #{index}')
    print([cv.get_feature_names_out()[i] for i in topic.argsort()[-n:]])

The top 15 words for topic #0
['california', 'america', 'home', 'hurricane', 'times', 'president', 'clinton', 'york', 'like', 'city', 'people', 'new', 'ms', 'mr', 'trump']
The top 15 words for topic #1
['states', 'money', 'mobil', 'tillerson', 'years', 'year', 'arabia', 'exxon', 'afghanistan', 'world', 'taliban', 'government', 'percent', 'saudi', 'mr']
The top 15 words for topic #2
['like', 'work', 'officials', 'state', 'cruz', 'government', 'did', 'flynn', 'dr', 'president', 'police', 'people', 'weight', 'trump', 'mr']
The top 15 words for topic #3
['years', 'islamic', 'party', 'people', 'united', 'isis', 'intelligence', 'country', 'speech', 'state', 'group', 'president', 'new', 'trump', 'mr']
The top 15 words for topic #4
['rate', 'fed', 'want', 'study', 'year', 'percent', 'ms', 'mr', 'state', 'like', 'new', 'tax', 'briefing', 'people', '_____']
The top 15 words for topic #5
['album', 'did', 'night', 'going', 'man', 'years', 'people', 'new', 'just', 'united', 'time', 'states', 'like'

Work on unseen article

In [41]:
new_text = ["The government is discussing new economic policies"]

# Transform using the SAME vectorizer used during training
X_new = cv.transform(new_text)

# Predict topic distribution
topic_distribution = lda.transform(X_new)

print(topic_distribution)

[[0.02386703 0.02392816 0.02387327 0.29685895 0.02393759 0.02393285
  0.58360216]]
